# `utils_catalogo.ipynb` - Utilidades del catalogo de ubicaciones

Componente de utilidades (independiente de extraccion, limpieza,
agregacion y visualizacion, tal y como pide el enunciado en "Estructura y
Principios de Diseno") con funciones auxiliares sobre el catalogo de
ciudades de referencia: validacion de que las coordenadas caen dentro del
territorio peninsular espanol, y una vista tabular del catalogo para
inspeccion y para las pruebas unitarias de `tests/test_catalogo.py`.

> **Decision de diseno**: el rango de referencia
> `(LAT_MIN_PENINSULA, LAT_MAX_PENINSULA, LON_MIN_PENINSULA, LON_MAX_PENINSULA)`
> es una caja delimitadora amplia de la Espana peninsular (excluye
> Canarias, incluye con margen Baleares), suficiente para detectar un
> error de tecleo en una coordenada del catalogo sin depender de una
> libreria de geometria pesada para un chequeo tan simple.

> Depende de: `config.ipynb`.

In [1]:
import os
if os.getcwd().endswith("climate_engine"):
    os.chdir("..")
%run climate_engine/config.ipynb

Fichero de configuracion: /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/config.yaml
Raiz del proyecto:        /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco
Ubicaciones cargadas (5): ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
Variables horarias:       ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
Ventana de prediccion:    14 dias, zona horaria UTC
Politica de reintentos:   3 reintentos, backoff x2, codigos [429, 500, 502, 503, 504]
Rango temperatura valido: (-50.0, 60.0)
Rango humedad valido:     (0.0, 100.0)
Ruta bronze:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/bronze  (existe: True)
Ruta silver:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/silver  (existe: True)
Ruta gold:    /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/gold  (existe: True)
URL base API: https://api.open-me

In [2]:
import pandas as pd

## Funcion principal

In [ ]:
# Caja delimitadora aproximada de la Espana peninsular.
LAT_MIN_PENINSULA, LAT_MAX_PENINSULA = 35.5, 44.0
LON_MIN_PENINSULA, LON_MAX_PENINSULA = -9.5, 3.5

CIUDADES_MINIMAS_REQUERIDAS = {"Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao"}


def coordenadas_en_peninsula(lat, lon):
    """Comprueba si una coordenada (lat, lon) cae dentro de la caja
    delimitadora de la Espana peninsular.

    Devuelve: booleano.
    """
    return (LAT_MIN_PENINSULA <= lat <= LAT_MAX_PENINSULA) and (
        LON_MIN_PENINSULA <= lon <= LON_MAX_PENINSULA
    )


def catalogo_como_dataframe(ubicaciones=None):
    """Convierte el catalogo de ubicaciones (dict ciudad -> {lat, lon}) en
    un DataFrame tabular, con una columna adicional indicando si la
    coordenada cae dentro de la Peninsula.

    Devuelve: DataFrame con columnas [ciudad, lat, lon, en_peninsula].
    """
    ubicaciones = ubicaciones if ubicaciones is not None else UBICACIONES
    filas = [
        {
            "ciudad": nombre,
            "lat": coords["lat"],
            "lon": coords["lon"],
            "en_peninsula": coordenadas_en_peninsula(coords["lat"], coords["lon"]),
        }
        for nombre, coords in ubicaciones.items()
    ]
    return pd.DataFrame(filas)


def validar_catalogo_minimo(ubicaciones=None):
    """Valida que el catalogo cumple el requisito del enunciado: al menos
    las cinco ciudades de referencia (Madrid, Barcelona, Valencia, Sevilla,
    Bilbao), todas con coordenadas dentro de la Peninsula.

    Lanza AssertionError si no se cumple. Devuelve True si es correcto.
    """
    ubicaciones = ubicaciones if ubicaciones is not None else UBICACIONES
    faltantes = CIUDADES_MINIMAS_REQUERIDAS - set(ubicaciones.keys())
    assert not faltantes, f"Faltan ciudades obligatorias en el catalogo: {faltantes}" # assert sirve para lanzar un error si la condición no se cumple

    df = catalogo_como_dataframe(ubicaciones)
    fuera_de_rango = df[~df["en_peninsula"]]
    assert fuera_de_rango.empty, (
        f"Coordenadas fuera de la Peninsula en el catalogo: "
        f"{fuera_de_rango['ciudad'].tolist()}"
    )
    return True

## Prueba rapida

Se vuelca el catalogo completo como tabla y se valida que cumple el requisito minimo del enunciado (cinco ciudades peninsulares).

In [4]:
df_catalogo = catalogo_como_dataframe()
print(df_catalogo.to_string(index=False))

validar_catalogo_minimo()
print("\nValidacion del catalogo minimo (5 ciudades peninsulares): OK")

   ciudad     lat     lon  en_peninsula
   Madrid 40.4168 -3.7038          True
Barcelona 41.3851  2.1734          True
 Valencia 39.4699 -0.3763          True
  Sevilla 37.3891 -5.9845          True
   Bilbao 43.2630 -2.9350          True

Validacion del catalogo minimo (5 ciudades peninsulares): OK


---
✅ **`utils_catalogo.ipynb` verificado/a.**